In [1]:
import numpy as np
import pandas as pd
from sklearn.utils import resample
import transformers
from transformers import AutoTokenizer, AutoModel
import torch.nn as nn
from peft import LoraConfig, get_peft_model, TaskType
from pyfaidx import Fasta
import pandas as pd
import torch
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, recall_score, roc_auc_score, confusion_matrix, f1_score
from sklearn.linear_model import LogisticRegression
from torch.utils.data import DataLoader, Subset
import torch.nn.functional as F
from sklearn.utils.class_weight import compute_class_weight
from xgboost import XGBClassifier

c:\Users\admin\anaconda3\envs\dnabert2_cftr\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
c:\Users\admin\anaconda3\envs\dnabert2_cftr\lib\site-packages\accelerate\utils\torch_xla.py:18: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


In [2]:
%pip install xgboost

In [3]:
genome = Fasta(r"D:\CFTR\Homo_sapiens_CFTR_sequence.fa")

In [4]:
genome = Fasta(r"Homo_sapiens_CFTR_sequence.fa")

df = pd.read_csv(r"final_cftr_dataset.csv")
df = df[~df["Variant type"].isin(["Haplotype"])]

df = df.sample(frac=1, random_state=42).reset_index(drop=True)

In [5]:
df_major = df[df["cause"] == 1]
df_minor = df[df["cause"] == 0]

df_minor_upsampled = resample(df_minor,
                             replace=True,
                             n_samples=len(df_major),
                             random_state=42)

df_balanced = pd.concat([df_major, df_minor_upsampled])

In [6]:
df_balanced.head()

,Variant cDNA name,variant determination,hgvs_genomic_grch38,chr,pos,ref,alt,cause,Name,Canonical SPDI,dbSNP ID,Variant type,Germline classification,Allele Frequency,ClinVar Germline Classification
0,c.4136+2T>G,CF-causing,NC_000007.14:g.117664862T>G,chr7,117664862,T,G,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,c.3397del,CF-causing,NC_000007.14:g.117614642del,chr7,117614640,TC,T,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,c.328G>C,CF-causing,NC_000007.14:g.117530953G>C,chr7,117530953,G,C,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,c.2299C>T,CF-causing,NC_000007.14:g.117592466C>T,chr7,117592466,C,T,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,c.3353C>T,CF-causing,NC_000007.14:g.117611794C>T,chr7,117611794,C,T,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [7]:
df_model = df_balanced[["chr", "pos", "ref", "alt", "cause"]]
df_model.rename(columns={"cause": "label"}, inplace=True)

C:\Users\admin\AppData\Local\Temp\ipykernel_9680\183857323.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_model.rename(columns={"cause": "label"}, inplace=True)


In [8]:
df_model = df_model.dropna(subset=["chr", "pos", "ref", "alt"])

In [9]:
df_model.head()

,chr,pos,ref,alt,label
0,chr7,117664862,T,G,1
1,chr7,117614640,TC,T,1
2,chr7,117530953,G,C,1
3,chr7,117592466,C,T,1
4,chr7,117611794,C,T,1


In [10]:
df_model["ref"] = df_model["ref"].astype(str)
df_model["alt"] = df_model["alt"].astype(str)
df_model["pos"] = df_model["pos"].astype(int)

In [11]:
df_model["chr"] = df_model["chr"].astype(str).str.replace("chr", "")

In [12]:
df_model = df_model[
    (df_model["ref"].str.len() > 0) &
    (df_model["alt"].str.len() > 0)
]

df_model.reset_index(drop=True, inplace=True)

In [13]:
def extract_window(seq, pos, window=50):
    start = max(0, pos - window)
    end = pos + window
    return seq[start:end]

In [14]:
def apply_mutation(seq, ref, alt, pos):
    if not isinstance(ref, str) or not isinstance(alt, str):
        return None
    return seq[:pos] + alt + seq[pos+len(ref):]

In [15]:
OFFSET = 117287120

def create_mut_seq(row, genome):
    try:
        chrom = row["chr"]
        pos = row["pos"] - OFFSET
        ref = row["ref"]
        alt = row["alt"]

        full_seq = str(genome[chrom])  # convert pyfaidx → string

        window_start = max(0, pos - 50)
        window_end = pos + 50

        window_seq = full_seq[window_start:window_end]

        rel_pos = pos - window_start

        mut_seq = apply_mutation(window_seq, ref, alt, rel_pos)

        return mut_seq

    except:
        return None

In [16]:
df_model["sequence"] = df_model.apply(
    lambda x: create_mut_seq(x, genome), axis=1
)

In [17]:
df_model.dropna(subset=["sequence"], inplace=True)

In [18]:
df_model.reset_index(drop=True, inplace=True)

In [19]:
def reverse_complement(seq):
    comp = {'A':'T','T':'A','C':'G','G':'C'}
    return ''.join(comp.get(b, 'N') for b in reversed(seq))

In [20]:
df_model["rc_sequence"] = df_model["sequence"].apply(reverse_complement)

In [21]:
df_model.head()

,chr,pos,ref,alt,label,sequence,rc_sequence
0,7,117664862,T,G,1,AGGCGAAGATCTTGCTGCTTGATGAACCCAGTGCTCATTTGGATCC...,AATGATTCTGTTCCCACTGTGCTATTAAGTAACAGAACATCTGAAA...
1,7,117614640,TC,T,1,TTACGTCTTTTGTGCATCTATAGGAGAAGGAGAAGGAAGAGTTGGT...,GTTTACAGCCCACTGCAATGTACTCATGATATTCATGGCTAAAGTC...
2,7,117530953,G,C,1,TCACCAAAGCAGTACAGCCTCTCTTACTGGGAAGAATCATAGCTTC...,CCTATGCCTAGATAAATCGCGATAGAGCGTTCCTCCTTGTTATCCG...
3,7,117592466,C,T,1,CTCGCATCAGCGTGATCAGCACTGGCCCCACGCTTCAGGCACGAAG...,ATGTTCTGACCTTGGTTAACTGAGTGTGTCATCAGGTTCAGGACAG...
4,7,117611794,C,T,1,GAGAATAGAAATGATTTTTGTCATCTTCTTCATTGCTGTTACCTTC...,TAAATGCTTAGCTAAAGTTAATGAGTTCATAGTACCTGTTGTTAAA...


In [22]:
#df_model.rename(columns={"cause": "label"}, inplace=True)

In [23]:
df_final = pd.concat([
    df_model[["sequence", "label"]],
    df_model[["rc_sequence", "label"]].rename(columns={"rc_sequence": "sequence"})
], ignore_index=True)

In [24]:
df_final.sample(10)

,sequence,label
1695,GTGAAGCTCTTTCCCCACCGGAACTCAAGCAAGTGCAAGTCTAAGC...,0
5776,AGCAAATGCTTGCTAGACCAATAATTAGTTATTCACCTTGCTAAAG...,0
4691,CTATCCACATCTATGCTGGAGTTTACAGCCCACTGCAATGTACTCA...,0
4177,ATACCTACCTTTAAGTCTTCTTCGTTAATTTCTTCACTTATTTCCA...,1
2041,TATCTGACAAACTCATCTTTTATTTTTGATGTGTGTGTGTGTGTGT...,0
1566,CCAAGTGACAAATAGCAAGTGTTGCATTTTACAAGTTATTTTTTAG...,0
5564,CAACATTTTTGTTTTTAAGAATGGCAGACAATTTCACAATTAGTTT...,0
5310,TAATTCCCCAAATCCCTGTTAAAAAAACACACACACACACACACAC...,0
5055,CCACTGTGCTTAATTTTACCCTCTGAAGGCTCCAGTTCTCCCATAA...,0
155,TGTTAAGGCATACTGCTGGGAAGAAGCAATGGAAAAAATGATTGAA...,1


In [25]:
X = df_final["sequence"].values
y = df_final["label"].values

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

In [26]:
model_name = "zhihan1996/DNABERT-2-117M"
tokenizer = AutoTokenizer.from_pretrained(
    model_name,
    trust_remote_code=True
)

model = AutoModel.from_pretrained(
    model_name,
    trust_remote_code=True
)

model.eval()

c:\Users\admin\anaconda3\envs\dnabert2_cftr\lib\site-packages\huggingface_hub\file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
Explicitly passing a `revision` is encouraged when loading a configuration with custom code to ensure no malicious code has been contributed in a newer revision.
Explicitly passing a `revision` is encouraged when loading a model with custom code to ensure no malicious code has been contributed in a newer revision.
C:\Users\admin/.cache\huggingface\modules\transformers_modules\zhihan1996\DNABERT-2-117M\7bce263b15377fc15361f52cfab88f8b586abda0\bert_layers.py:126: UserWarning: Unable to import Triton; defaulting MosaicBERT attention implementation to pytorch (this will reduce throughput when using this model).
  warnings.warn(
Some weights of the model checkpoint at zhihan1996/DNABERT-2-11

BertModel(
  (embeddings): BertEmbeddings(
    (word_embeddings): Embedding(4096, 768)
    (token_type_embeddings): Embedding(2, 768)
    (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
    (dropout): Dropout(p=0.1, inplace=False)
  )
  (encoder): BertEncoder(
    (layer): ModuleList(
      (0-11): 12 x BertLayer(
        (attention): BertUnpadAttention(
          (self): BertUnpadSelfAttention(
            (dropout): Dropout(p=0.0, inplace=False)
            (Wqkv): Linear(in_features=768, out_features=2304, bias=True)
          )
          (output): BertSelfOutput(
            (dense): Linear(in_features=768, out_features=768, bias=True)
            (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
        )
        (mlp): BertGatedLinearUnitMLP(
          (gated_layers): Linear(in_features=768, out_features=6144, bias=False)
          (act): GELU(approximate='none')
          (wo): L

In [27]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))
model.to(device)


Using device: cuda
True
NVIDIA RTX A5000


BertModel(
  (embeddings): BertEmbeddings(
    (word_embeddings): Embedding(4096, 768)
    (token_type_embeddings): Embedding(2, 768)
    (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
    (dropout): Dropout(p=0.1, inplace=False)
  )
  (encoder): BertEncoder(
    (layer): ModuleList(
      (0-11): 12 x BertLayer(
        (attention): BertUnpadAttention(
          (self): BertUnpadSelfAttention(
            (dropout): Dropout(p=0.0, inplace=False)
            (Wqkv): Linear(in_features=768, out_features=2304, bias=True)
          )
          (output): BertSelfOutput(
            (dense): Linear(in_features=768, out_features=768, bias=True)
            (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
        )
        (mlp): BertGatedLinearUnitMLP(
          (gated_layers): Linear(in_features=768, out_features=6144, bias=False)
          (act): GELU(approximate='none')
          (wo): L

In [28]:
def get_embedding_batch(sequences, batch_size=32):
    embeddings = []

    for i in range(0, len(sequences), batch_size):
        batch = sequences[i:i+batch_size]

        inputs = tokenizer(
            list(batch),
            return_tensors="pt",
            padding=True,
            truncation=True
        )

        inputs = {k: v.to(device) for k, v in inputs.items()}

        with torch.no_grad():
            outputs = model(**inputs)

        cls_embeddings = outputs[0][:, 0, :]

        embeddings.append(cls_embeddings.cpu().numpy())

    return np.vstack(embeddings)

In [29]:
X_train_emb = get_embedding_batch(X_train)
X_test_emb  = get_embedding_batch(X_test)

Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.


In [30]:
lr = LogisticRegression(max_iter=1000)
lr.fit(X_train_emb, y_train)

y_pred_lr = lr.predict(X_test_emb)

In [35]:
print(torch.cuda.is_available())

True


In [38]:
xgb = XGBClassifier(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    tree_method="hist",
)

In [39]:
xgb.fit(X_train_emb, y_train)

y_pred_xgb = xgb.predict(X_test_emb)

In [40]:
from sklearn.metrics import classification_report, roc_auc_score

print("XGBoost Results")
print(classification_report(y_test, y_pred_xgb))

print("ROC-AUC:",
      roc_auc_score(y_test, xgb.predict_proba(X_test_emb)[:,1]))

XGBoost Results
              precision    recall  f1-score   support

           0       1.00      1.00      1.00       625
           1       1.00      0.99      1.00       588

    accuracy                           1.00      1213
   macro avg       1.00      1.00      1.00      1213
weighted avg       1.00      1.00      1.00      1213

ROC-AUC: 0.9999972789115645
